# 01 — Data Preprocessing for AdaFuse

**Project:** AdaFuse — Adaptive Attention Fusion of Sequential and Collaborative Representations for Recommendation  
**BTP, IIT Patna — 2026**

This notebook preprocesses two datasets for the AdaFuse experiments:

| Dataset | Source Format | Users | Items | Interactions | Timestamps? | Metadata? |
|---------|-------------|-------|-------|-------------|-------------|----------|
| **Amazon-Book** | LightGCN-format text (pre-split) | ~52K | ~92K | ~3M | No | No |
| **Amazon-Beauty** | Raw JSONL reviews + Parquet metadata | ~632K raw | ~113K raw | ~701K raw | Yes | Yes |

### What this notebook produces

For each dataset we produce:
1. Train / validation / test interaction dicts (`{user_id: [item_ids]}`)
2. A normalised sparse adjacency matrix built **from training data only**
3. User interaction sequences for SASRec (time-ordered for Beauty, list-ordered for Book)
4. Negative-sampling utilities for BPR training
5. (Beauty only) Item metadata mapping (category, brand)

### Key invariants (from AGENTS.md)

- Build the recommendation graph from **training interactions only**
- Offset item node IDs by `num_users` so the graph is over `num_users + num_items` nodes
- Add **both** edge directions → `edge_index.shape == [2, 2 * num_train_interactions]`
- Normalise: `D^{-1/2} A D^{-1/2}`
- Embedding dim = 64, seeds = `[42, 123, 456, 789, 1024]`

---
## 1. Setup & Imports

In [ ]:
import os
import json
import pickle
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt

# Reproducibility
SEEDS = [42, 123, 456, 789, 1024]
SEED = SEEDS[0]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths — all relative to the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
ROADMAP_DATA = os.path.join(PROJECT_ROOT, 'roadmap', 'data')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Roadmap data : {ROADMAP_DATA}')
print(f'Processed dir: {PROCESSED_DIR}')

---
## 2. Amazon-Book Dataset

The Amazon-Book files are already in **LightGCN format**:  
each line is `user_id  item_id  item_id  item_id ...` (space-separated).  
There is a `train.txt` and a `test.txt` but **no validation file and no timestamps**.

We will:
1. Parse both files into `{user_id: [item_ids]}` dicts.
2. Create a validation set by randomly holding out one training item per user (for users with ≥3 training items).
3. Build the bipartite graph from the remaining training interactions.

In [ ]:
def load_lightgcn_file(path):
    """Parse a LightGCN-format file into {user_id: [item_ids]}."""
    data = {}
    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            user = int(parts[0])
            items = [int(x) for x in parts[1:]]
            data[user] = items
    return data


book_train_path = os.path.join(ROADMAP_DATA, 'amazon-book', 'train.txt')
book_test_path  = os.path.join(ROADMAP_DATA, 'amazon-book', 'test.txt')

book_train_raw = load_lightgcn_file(book_train_path)
book_test_raw  = load_lightgcn_file(book_test_path)

print(f'Loaded Amazon-Book  train: {len(book_train_raw)} users')
print(f'Loaded Amazon-Book  test : {len(book_test_raw)} users')

### 2.1 Basic statistics

In [ ]:
all_users_book = set(book_train_raw.keys()) | set(book_test_raw.keys())
all_items_book = set()
for items in book_train_raw.values():
    all_items_book.update(items)
for items in book_test_raw.values():
    all_items_book.update(items)

num_users_book = len(all_users_book)
num_items_book = max(all_items_book) + 1          # 0-indexed
num_train_book = sum(len(v) for v in book_train_raw.values())
num_test_book  = sum(len(v) for v in book_test_raw.values())
density_book   = num_train_book / (num_users_book * num_items_book)

print('=== Amazon-Book Statistics ===')
print(f'  Users           : {num_users_book:,}')
print(f'  Items (max id+1): {num_items_book:,}')
print(f'  Train interact. : {num_train_book:,}')
print(f'  Test interact.  : {num_test_book:,}')
print(f'  Density         : {density_book:.6f}')

### 2.2 Create a validation split

Since Amazon-Book has no timestamps and no validation file, we randomly hold out **one item per user** from the training set for users who have **≥ 3 training items** (so training still has ≥ 2 items for sequence modelling).

In [ ]:
rng = np.random.RandomState(SEED)

book_train = {}
book_val   = {}

for user, items in book_train_raw.items():
    if len(items) >= 3:
        val_idx = rng.randint(0, len(items))
        val_item = items[val_idx]
        train_items = items[:val_idx] + items[val_idx + 1:]
        book_train[user] = train_items
        book_val[user] = [val_item]
    else:
        book_train[user] = items
        # no validation item for very sparse users

book_test = book_test_raw  # unchanged

print(f'Train users: {len(book_train):,}  |  interactions: {sum(len(v) for v in book_train.values()):,}')
print(f'Val   users: {len(book_val):,}  |  interactions: {sum(len(v) for v in book_val.values()):,}')
print(f'Test  users: {len(book_test):,}  |  interactions: {sum(len(v) for v in book_test.values()):,}')

### 2.3 Build interaction sequences for SASRec

Amazon-Book has **no timestamps**, so we treat the item list order in the training file as an implicit sequence (this is standard practice in the literature when timestamps are unavailable). We truncate to a maximum length of 50.

In [ ]:
MAX_SEQ_LEN = 50

def build_sequences(user_items, max_len=MAX_SEQ_LEN):
    """Truncate each user's item list to the most recent `max_len` items."""
    sequences = {}
    for user, items in user_items.items():
        sequences[user] = items[-max_len:]  # keep last max_len
    return sequences

book_sequences = build_sequences(book_train)

seq_lens = [len(s) for s in book_sequences.values()]
print(f'Sequence stats: min={min(seq_lens)}, max={max(seq_lens)}, '
      f'mean={np.mean(seq_lens):.1f}, median={np.median(seq_lens):.1f}')

---
## 3. Amazon-Beauty Dataset

The raw data is a JSONL file with fields:
- `user_id` (string)
- `parent_asin` (string — item identifier)
- `rating` (float)
- `timestamp` (int, epoch milliseconds)

We will:
1. Load and explore the data
2. Apply **10-core filtering** (remove users/items with < 10 interactions, iteratively until stable)
3. Re-index to contiguous 0-based integer IDs
4. Apply **leave-one-out** temporal split (last → test, second-to-last → validation, rest → train)
5. Load item metadata from the Parquet file

### 3.1 Load raw data

In [ ]:
beauty_jsonl_path = os.path.join(
    ROADMAP_DATA, 'amazon-beauty', 'raw', 'review_categories', 'All_Beauty.jsonl'
)

beauty_raw = pd.read_json(beauty_jsonl_path, lines=True)
print(f'Raw rows: {len(beauty_raw):,}')
print(f'Columns : {list(beauty_raw.columns)}')
beauty_raw.head(3)

In [ ]:
# Keep only the columns we need
beauty_df = beauty_raw[['user_id', 'parent_asin', 'rating', 'timestamp']].copy()
beauty_df.columns = ['user', 'item', 'rating', 'timestamp']

# Convert to implicit feedback: any interaction counts
beauty_df = beauty_df.drop_duplicates(subset=['user', 'item'])

print(f'After dedup: {len(beauty_df):,} interactions')
print(f'Unique users: {beauty_df["user"].nunique():,}')
print(f'Unique items: {beauty_df["item"].nunique():,}')
print()
print('Rating distribution:')
print(beauty_df['rating'].value_counts().sort_index())

### 3.2 K-core filtering (10-core)

Iteratively remove users and items with fewer than 10 interactions until convergence. This ensures both the Transformer and GNN branches have enough signal per entity.

In [ ]:
def k_core_filter(df, k=10, user_col='user', item_col='item'):
    """Iteratively remove users/items with < k interactions until stable."""
    prev_len = 0
    iteration = 0
    while len(df) != prev_len:
        prev_len = len(df)
        iteration += 1
        # Filter users
        user_counts = df[user_col].value_counts()
        valid_users = user_counts[user_counts >= k].index
        df = df[df[user_col].isin(valid_users)]
        # Filter items
        item_counts = df[item_col].value_counts()
        valid_items = item_counts[item_counts >= k].index
        df = df[df[item_col].isin(valid_items)]
        print(f'  Iteration {iteration}: {len(df):,} interactions, '
              f'{df[user_col].nunique():,} users, {df[item_col].nunique():,} items')
    return df.copy()

print('Applying 10-core filtering...')
beauty_filtered = k_core_filter(beauty_df, k=10)

### 3.3 Re-index IDs

Map the original string user/item IDs to contiguous 0-based integers.

In [ ]:
# Build mappings
unique_users_beauty = beauty_filtered['user'].unique()
unique_items_beauty = beauty_filtered['item'].unique()

user2id_beauty = {u: i for i, u in enumerate(sorted(unique_users_beauty))}
item2id_beauty = {it: i for i, it in enumerate(sorted(unique_items_beauty))}

# Store reverse maps for later metadata joining
id2item_beauty = {v: k for k, v in item2id_beauty.items()}

beauty_filtered = beauty_filtered.copy()
beauty_filtered['user_idx'] = beauty_filtered['user'].map(user2id_beauty)
beauty_filtered['item_idx'] = beauty_filtered['item'].map(item2id_beauty)

num_users_beauty = len(user2id_beauty)
num_items_beauty = len(item2id_beauty)
num_interactions_beauty = len(beauty_filtered)

print(f'After 10-core + re-indexing:')
print(f'  Users       : {num_users_beauty:,}')
print(f'  Items       : {num_items_beauty:,}')
print(f'  Interactions: {num_interactions_beauty:,}')
print(f'  Density     : {num_interactions_beauty / (num_users_beauty * num_items_beauty):.6f}')

### 3.4 Temporal leave-one-out split

For each user, sort interactions by timestamp, then:
- **Last** item → test
- **Second-to-last** → validation
- **Rest** → training

In [ ]:
# Sort globally by user then timestamp
beauty_sorted = beauty_filtered.sort_values(['user_idx', 'timestamp'])

beauty_train_dict = defaultdict(list)
beauty_val_dict   = {}
beauty_test_dict  = {}

for user_idx, group in beauty_sorted.groupby('user_idx'):
    items = group['item_idx'].tolist()
    if len(items) < 3:
        # Not enough for 3-way split; put all in training
        beauty_train_dict[user_idx] = items
        continue
    beauty_test_dict[user_idx]  = [items[-1]]
    beauty_val_dict[user_idx]   = [items[-2]]
    beauty_train_dict[user_idx] = items[:-2]

beauty_train_dict = dict(beauty_train_dict)

n_train = sum(len(v) for v in beauty_train_dict.values())
n_val   = sum(len(v) for v in beauty_val_dict.values())
n_test  = sum(len(v) for v in beauty_test_dict.values())

print(f'Beauty split:')
print(f'  Train: {len(beauty_train_dict):,} users, {n_train:,} interactions')
print(f'  Val  : {len(beauty_val_dict):,} users, {n_val:,} interactions')
print(f'  Test : {len(beauty_test_dict):,} users, {n_test:,} interactions')

### 3.5 Build interaction sequences (time-ordered)

For SASRec we need each user's item sequence sorted by time. These come directly from the training split (which is already time-sorted).

In [ ]:
beauty_sequences = build_sequences(beauty_train_dict, max_len=MAX_SEQ_LEN)

seq_lens_b = [len(s) for s in beauty_sequences.values()]
print(f'Beauty sequence stats: min={min(seq_lens_b)}, max={max(seq_lens_b)}, '
      f'mean={np.mean(seq_lens_b):.1f}, median={np.median(seq_lens_b):.1f}')

### 3.6 Load item metadata

The Parquet file contains item metadata including category and brand. We map it to our new integer item IDs.

In [ ]:
meta_path = os.path.join(
    ROADMAP_DATA, 'amazon-beauty', 'raw_meta_All_Beauty', 'full-00000-of-00001.parquet'
)

meta_df = pd.read_parquet(meta_path)
print(f'Metadata rows: {len(meta_df):,}')
print(f'Columns: {list(meta_df.columns)}')
meta_df.head(3)

In [ ]:
# Identify the ASIN column — typically 'parent_asin' or 'asin'
asin_col = 'parent_asin' if 'parent_asin' in meta_df.columns else 'asin'

# Keep only items that survived filtering
meta_relevant = meta_df[meta_df[asin_col].isin(item2id_beauty)].copy()
meta_relevant['item_idx'] = meta_relevant[asin_col].map(item2id_beauty)

# Extract category and brand (handle list-type categories)
def extract_main_category(cats):
    if isinstance(cats, list) and len(cats) > 0:
        return cats[0]
    if isinstance(cats, str) and cats:
        return cats
    return 'Unknown'

cat_col = None
for c in ['main_category', 'categories', 'category']:
    if c in meta_relevant.columns:
        cat_col = c
        break

brand_col = 'brand' if 'brand' in meta_relevant.columns else None

item_metadata = {}
for _, row in meta_relevant.iterrows():
    idx = row['item_idx']
    category = extract_main_category(row[cat_col]) if cat_col else 'Unknown'
    brand    = row[brand_col] if (brand_col and pd.notna(row[brand_col])) else 'Unknown'
    item_metadata[idx] = {'category': category, 'brand': brand}

print(f'Metadata mapped for {len(item_metadata):,} / {num_items_beauty:,} items')
# Show a few examples
for k in list(item_metadata.keys())[:5]:
    print(f'  item {k}: {item_metadata[k]}')

---
## 4. Graph Construction Utility

Build the normalised bipartite adjacency matrix from training interactions.

**Procedure:**
1. Offset all item IDs by `num_users` so the full graph has `num_users + num_items` nodes.
2. Add edges in **both** directions (user→item and item→user).
3. Normalise: `Ã = D^{-1/2} A D^{-1/2}`.
4. Convert to a PyTorch sparse tensor for efficient GNN propagation.

In [ ]:
def build_adj_matrix(train_dict, num_users, num_items):
    """
    Build a normalised symmetric adjacency matrix for LightGCN.

    Returns
    -------
    adj_norm : scipy.sparse.csr_matrix
        Normalised adjacency of shape (num_users+num_items, num_users+num_items).
    edge_index : np.ndarray of shape (2, 2*num_interactions)
        COO-format edge list with both directions.
    """
    rows, cols = [], []
    for user, items in train_dict.items():
        for item in items:
            rows.append(user)
            cols.append(item + num_users)  # offset item IDs

    num_interactions = len(rows)
    n = num_users + num_items

    # user -> item edges
    row_np = np.array(rows, dtype=np.int64)
    col_np = np.array(cols, dtype=np.int64)

    # Both directions
    bi_rows = np.concatenate([row_np, col_np])
    bi_cols = np.concatenate([col_np, row_np])
    vals    = np.ones(len(bi_rows), dtype=np.float32)

    adj = sp.csr_matrix((vals, (bi_rows, bi_cols)), shape=(n, n))

    # edge_index for PyG compatibility
    edge_index = np.stack([bi_rows, bi_cols], axis=0)

    # Normalise: D^{-1/2} A D^{-1/2}
    degrees = np.array(adj.sum(axis=1)).flatten()
    d_inv_sqrt = np.power(degrees, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    D_inv_sqrt = sp.diags(d_inv_sqrt)
    adj_norm = D_inv_sqrt @ adj @ D_inv_sqrt

    return adj_norm, edge_index, num_interactions


def sparse_scipy_to_torch(adj_sp):
    """Convert a scipy sparse matrix to a PyTorch sparse tensor."""
    coo = adj_sp.tocoo()
    indices = torch.LongTensor(np.stack([coo.row, coo.col], axis=0))
    values  = torch.FloatTensor(coo.data)
    shape   = torch.Size(coo.shape)
    return torch.sparse_coo_tensor(indices, values, shape).coalesce()

### 4.1 Build Amazon-Book graph

In [ ]:
book_adj_norm, book_edge_index, book_num_train = build_adj_matrix(
    book_train, num_users_book, num_items_book
)
book_adj_torch = sparse_scipy_to_torch(book_adj_norm)

print('=== Amazon-Book Graph ===')
print(f'  Adjacency shape : {book_adj_norm.shape}')
print(f'  edge_index shape: {book_edge_index.shape}')
print(f'  Train interact. : {book_num_train:,}')
print(f'  Total edges     : {book_edge_index.shape[1]:,}  (expected 2 x {book_num_train:,} = {2*book_num_train:,})')
assert book_edge_index.shape == (2, 2 * book_num_train), 'Edge count mismatch!'
print('  ✓ Assertion passed: edge_index.shape == [2, 2 * num_train_interactions]')

### 4.2 Build Amazon-Beauty graph

In [ ]:
beauty_adj_norm, beauty_edge_index, beauty_num_train = build_adj_matrix(
    beauty_train_dict, num_users_beauty, num_items_beauty
)
beauty_adj_torch = sparse_scipy_to_torch(beauty_adj_norm)

print('=== Amazon-Beauty Graph ===')
print(f'  Adjacency shape : {beauty_adj_norm.shape}')
print(f'  edge_index shape: {beauty_edge_index.shape}')
print(f'  Train interact. : {beauty_num_train:,}')
print(f'  Total edges     : {beauty_edge_index.shape[1]:,}  (expected 2 x {beauty_num_train:,} = {2*beauty_num_train:,})')
assert beauty_edge_index.shape == (2, 2 * beauty_num_train), 'Edge count mismatch!'
print('  ✓ Assertion passed: edge_index.shape == [2, 2 * num_train_interactions]')

---
## 5. Negative Sampling Utility

For BPR training we need to sample negative items (items the user has **not** interacted with). We build a set of all interacted items per user and sample uniformly from the complement.

In [ ]:
def build_interaction_set(train_dict, val_dict=None, test_dict=None):
    """
    Build {user: set(all_interacted_items)} across all splits.
    Used to mask positives during negative sampling and evaluation.
    """
    all_items = defaultdict(set)
    for d in [train_dict, val_dict, test_dict]:
        if d is None:
            continue
        for user, items in d.items():
            all_items[user].update(items)
    return dict(all_items)


def sample_negative(user, interacted_set, num_items, rng=None):
    """Sample one item uniformly that the user has NOT interacted with."""
    if rng is None:
        rng = np.random.default_rng()
    while True:
        neg = rng.integers(0, num_items)
        if neg not in interacted_set:
            return neg


# Build interaction sets
book_all_items   = build_interaction_set(book_train, book_val, book_test)
beauty_all_items = build_interaction_set(beauty_train_dict, beauty_val_dict, beauty_test_dict)

print(f'Book   — users with interaction sets: {len(book_all_items):,}')
print(f'Beauty — users with interaction sets: {len(beauty_all_items):,}')

---
## 6. Dataset Statistics Summary

In [ ]:
print('╔════════════════════════════════════════════════════════════════╗')
print('║              Dataset Comparison Summary                       ║')
print('╠════════════════════════════════════════════════════════════════╣')
print(f'║ {"":30s} {"Amazon-Book":>14s} {"Amazon-Beauty":>14s} ║')
print(f'╠════════════════════════════════════════════════════════════════╣')
print(f'║ {"Users":30s} {num_users_book:>14,} {num_users_beauty:>14,} ║')
print(f'║ {"Items":30s} {num_items_book:>14,} {num_items_beauty:>14,} ║')
n_tr_book = sum(len(v) for v in book_train.values())
n_tr_beauty = sum(len(v) for v in beauty_train_dict.values())
print(f'║ {"Train interactions":30s} {n_tr_book:>14,} {n_tr_beauty:>14,} ║')
n_v_book = sum(len(v) for v in book_val.values())
n_v_beauty = sum(len(v) for v in beauty_val_dict.values())
print(f'║ {"Val interactions":30s} {n_v_book:>14,} {n_v_beauty:>14,} ║')
n_te_book = sum(len(v) for v in book_test.values())
n_te_beauty = sum(len(v) for v in beauty_test_dict.values())
print(f'║ {"Test interactions":30s} {n_te_book:>14,} {n_te_beauty:>14,} ║')
print(f'║ {"Graph nodes":30s} {num_users_book+num_items_book:>14,} {num_users_beauty+num_items_beauty:>14,} ║')
print(f'║ {"Graph edges (2×train)":30s} {2*book_num_train:>14,} {2*beauty_num_train:>14,} ║')
print(f'║ {"Timestamps available":30s} {"No":>14s} {"Yes":>14s} ║')
print(f'║ {"Item metadata available":30s} {"No":>14s} {"Yes":>14s} ║')
print('╚════════════════════════════════════════════════════════════════╝')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Dataset Statistics', fontsize=14, fontweight='bold')

# --- Amazon-Book ---
book_user_counts = [len(v) for v in book_train.values()]
book_item_counts_dict = defaultdict(int)
for items in book_train.values():
    for it in items:
        book_item_counts_dict[it] += 1
book_item_counts = list(book_item_counts_dict.values())

axes[0, 0].hist(book_user_counts, bins=50, color='#7c3aed', alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0, 0].set_title('Book: Interactions per User')
axes[0, 0].set_xlabel('# interactions')
axes[0, 0].set_ylabel('# users')
axes[0, 0].set_yscale('log')

axes[0, 1].hist(book_item_counts, bins=50, color='#00d4ff', alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0, 1].set_title('Book: Interactions per Item')
axes[0, 1].set_xlabel('# interactions')
axes[0, 1].set_ylabel('# items')
axes[0, 1].set_yscale('log')

book_seq_lens = [len(s) for s in book_sequences.values()]
axes[0, 2].hist(book_seq_lens, bins=50, color='#10b981', alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0, 2].set_title('Book: Sequence Lengths')
axes[0, 2].set_xlabel('sequence length')
axes[0, 2].set_ylabel('# users')

# --- Amazon-Beauty ---
beauty_user_counts = [len(v) for v in beauty_train_dict.values()]
beauty_item_counts_dict = defaultdict(int)
for items in beauty_train_dict.values():
    for it in items:
        beauty_item_counts_dict[it] += 1
beauty_item_counts = list(beauty_item_counts_dict.values())

axes[1, 0].hist(beauty_user_counts, bins=50, color='#7c3aed', alpha=0.8, edgecolor='white', linewidth=0.5)
axes[1, 0].set_title('Beauty: Interactions per User')
axes[1, 0].set_xlabel('# interactions')
axes[1, 0].set_ylabel('# users')
axes[1, 0].set_yscale('log')

axes[1, 1].hist(beauty_item_counts, bins=50, color='#00d4ff', alpha=0.8, edgecolor='white', linewidth=0.5)
axes[1, 1].set_title('Beauty: Interactions per Item')
axes[1, 1].set_xlabel('# interactions')
axes[1, 1].set_ylabel('# items')
axes[1, 1].set_yscale('log')

beauty_seq_lens = [len(s) for s in beauty_sequences.values()]
axes[1, 2].hist(beauty_seq_lens, bins=50, color='#10b981', alpha=0.8, edgecolor='white', linewidth=0.5)
axes[1, 2].set_title('Beauty: Sequence Lengths')
axes[1, 2].set_xlabel('sequence length')
axes[1, 2].set_ylabel('# users')

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, '..', '..', 'results', 'figures', 'dataset_stats.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure to results/figures/dataset_stats.png')

---
## 7. Save Everything

We save each dataset as a single pickle dictionary containing all preprocessed artifacts.

In [ ]:
# === Amazon-Book ===
book_data = {
    'num_users': num_users_book,
    'num_items': num_items_book,
    'train_dict': book_train,
    'val_dict': book_val,
    'test_dict': book_test,
    'sequences': book_sequences,
    'adj_norm': book_adj_norm,         # scipy sparse
    'adj_torch': book_adj_torch,       # torch sparse
    'edge_index': book_edge_index,     # numpy (2, 2E)
    'all_interacted': book_all_items,  # {user: set(items)}
}

book_out = os.path.join(PROCESSED_DIR, 'amazon_book_processed.pkl')
with open(book_out, 'wb') as f:
    pickle.dump(book_data, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Saved Amazon-Book  → {book_out}  ({os.path.getsize(book_out)/1e6:.1f} MB)')


# === Amazon-Beauty ===
beauty_data = {
    'num_users': num_users_beauty,
    'num_items': num_items_beauty,
    'train_dict': beauty_train_dict,
    'val_dict': beauty_val_dict,
    'test_dict': beauty_test_dict,
    'sequences': beauty_sequences,
    'adj_norm': beauty_adj_norm,
    'adj_torch': beauty_adj_torch,
    'edge_index': beauty_edge_index,
    'all_interacted': beauty_all_items,
    'item_metadata': item_metadata,       # {item_idx: {category, brand}}
    'user2id': user2id_beauty,
    'item2id': item2id_beauty,
    'id2item': id2item_beauty,
}

beauty_out = os.path.join(PROCESSED_DIR, 'amazon_beauty_processed.pkl')
with open(beauty_out, 'wb') as f:
    pickle.dump(beauty_data, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Saved Amazon-Beauty → {beauty_out}  ({os.path.getsize(beauty_out)/1e6:.1f} MB)')

---
## 8. Quick sanity check: reload and verify

In [ ]:
# Reload and verify
with open(book_out, 'rb') as f:
    check = pickle.load(f)

assert check['num_users'] == num_users_book
assert check['num_items'] == num_items_book
assert check['edge_index'].shape[0] == 2
print('✓ Amazon-Book reload OK')

with open(beauty_out, 'rb') as f:
    check = pickle.load(f)

assert check['num_users'] == num_users_beauty
assert check['num_items'] == num_items_beauty
assert check['edge_index'].shape[0] == 2
print('✓ Amazon-Beauty reload OK')
print()
print('Data preprocessing complete. Ready for model training.')